In [1]:
import pandas as pd
from pathlib import Path

# Reads the raw file created by the previous script
RAW_FILE = Path("../../tables/all_runs_raw.csv")

df = pd.read_csv(RAW_FILE)

# Safety
df = df[df["set"].isin(["set1", "set2"])].copy()

# Average per method/config/set
summary = (
    df.groupby(["set", "config", "method"], as_index=False)
    .agg(
        avg_objective=("objective", "mean"),
        best_objective=("objective", "min"),
        avg_runtime=("runtime", "mean"),
        runs=("objective", "count"),
    )
)

# Decide winners per set/config
rows = []

for (set_name, config), sub in summary.groupby(["set", "config"]):
    obj_winner = sub.loc[sub["avg_objective"].idxmin()]
    time_winner = sub.loc[sub["avg_runtime"].idxmin()]

    rows.append({
        "set": set_name,
        "config": config,
        "best_avg_objective_algorithm": obj_winner["method"],
        "best_avg_objective": obj_winner["avg_objective"],
        "fastest_algorithm": time_winner["method"],
        "best_avg_runtime": time_winner["avg_runtime"],
    })

comparison = pd.DataFrame(rows)

print("=== Summary by method ===")
display(summary.sort_values(["set", "config", "method"]))

print("=== Winners by set/config ===")
display(comparison.sort_values(["set", "config"]))

# Save results
OUTPUT_DIR = Path("../../tables")
summary.to_csv(OUTPUT_DIR / "algorithm_summary_by_config.csv", index=False)
comparison.to_csv(OUTPUT_DIR / "algorithm_winners_by_config.csv", index=False)

print("Saved:")
print(OUTPUT_DIR / "algorithm_summary_by_config.csv")
print(OUTPUT_DIR / "algorithm_winners_by_config.csv")

=== Summary by method ===


,set,config,method,avg_objective,best_objective,avg_runtime,runs
0,set1,set1_3types_linear,sfc,76829.266667,75170.0,9.209195,300
1,set1,set1_3types_linear,vanilla,76791.200000,75140.0,6.239323,300
2,set1,set1_5types_linear,sfc,76427.600000,75040.0,10.600247,300
3,set1,set1_5types_linear,vanilla,76427.800000,75050.0,11.175568,300
4,set2,set2_7types_concave,sfc,84337.880000,79583.0,9.448777,300
5,set2,set2_7types_concave,vanilla,83895.650000,79236.0,9.918457,300
6,set2,set2_7types_convex,sfc,179944.016667,175165.0,17.219918,300
7,set2,set2_7types_convex,vanilla,179877.496667,175146.0,16.843355,300
8,set2,set2_7types_linear,sfc,128303.866667,123730.0,10.673045,300
9,set2,set2_7types_linear,vanilla,128217.133333,123580.0,12.029427,300


=== Winners by set/config ===


,set,config,best_avg_objective_algorithm,best_avg_objective,fastest_algorithm,best_avg_runtime
0,set1,set1_3types_linear,vanilla,76791.200000,vanilla,6.239323
1,set1,set1_5types_linear,sfc,76427.600000,sfc,10.600247
2,set2,set2_7types_concave,vanilla,83895.650000,sfc,9.448777
3,set2,set2_7types_convex,vanilla,179877.496667,vanilla,16.843355
4,set2,set2_7types_linear,vanilla,128217.133333,sfc,10.673045


Saved:
../../tables/algorithm_summary_by_config.csv
../../tables/algorithm_winners_by_config.csv
